In [1]:
#import libraries
import os
import numpy as np
import pandas as pd
import timeit

from sklearn.model_selection import train_test_split, RepeatedKFold, GroupKFold
from sklearn.linear_model import Ridge, RidgeCV, LassoCV, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import geopandas as gpd
import seaborn as sns
import matplotlib.cm
import matplotlib.pyplot as plt
%matplotlib inline

## import MOSAIKS features

In [2]:
# import the first file 
df1 = pd.read_pickle(r'C:\HDRO\Stats\Stats2\MOSAIKS\DHS_DHS_dense_DHSID.p')
df1['ID']= df1['DHSID'].str.slice(0,2)
df1['year']= df1['DHSID'].str.slice(2,6)

In [3]:
# import the additinal file with 8 countries 
df2 = pd.read_pickle(r'C:\HDRO\Stats\Stats2\MOSAIKS\DHS_additional_global_dense_DHSID.p')
df2.columns = df2.columns.str.replace('dhsid', 'DHSID')
df2['ID']= df2['DHSID'].str.slice(0,2)
df2['year']= df2['DHSID'].str.slice(2,6)

In [4]:
# merge two datasets - this is the final dataset
df3 = pd.concat([df1, df2])
df4 = df3.drop(columns=['continent', 'ID', 'year'])

## Import NL features

In [5]:
nl = pd.read_pickle("C:/HDRO/Stats/Stats2/MOSAIKS/all_dhs_dmsp_nightlight_features_20_bins_GPW_pop_weighted.p")

In [6]:
nl

,perc_pixels_in_bin_0,perc_pixels_in_bin_1,perc_pixels_in_bin_2,perc_pixels_in_bin_3,perc_pixels_in_bin_4,perc_pixels_in_bin_5,perc_pixels_in_bin_6,perc_pixels_in_bin_7,perc_pixels_in_bin_8,perc_pixels_in_bin_9,perc_pixels_in_bin_10,perc_pixels_in_bin_11,perc_pixels_in_bin_12,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19
DHSID,,,,,,,,,,,,,,,,,,,,
CM201800000018,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
EG201401460501,0.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GN201800000186,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KE201400000266,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
PK201700000094,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RW201900000494,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RW201900000495,0.0,0.834223,0.038776,0.048623,0.041006,0.024467,0.012905,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RW201900000496,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Import income

In [7]:
# import income file
df_list = []
direct2 = ("C:/HDRO/Stats/Stats2/geo-inc/")
for file in os.listdir(direct2 + 'geo-income_module'):
    d = pd.read_stata(direct2 + 'geo-income_module/' + file)
    df_list.append(d)
df_inc = pd.concat(df_list)

In [8]:
income = df_inc.groupby("DHSID")[["lognormal_inc_ind_wiid"]].agg(np.nanmean).dropna()

In [9]:
income

,lognormal_inc_ind_wiid
DHSID,
AL201700000001,18942.701172
AL201700000002,18275.437500
AL201700000003,19139.130859
AL201700000004,20030.707031
AL201700000005,22517.060547
...,...
ZM201800000541,7729.175293
ZM201800000542,894.005615
ZM201800000543,3261.351807


## Import MPI data

In [10]:
#mpi file
direct = ("C:/HDRO/Stats/Stats2/MOSAIKS/")
df_mpi = pd.read_stata(direct + "mpi_geo37.dta")

In [11]:
dep_score = df_mpi.groupby("DHSID")[["weighted_deprivation_score"]].agg(np.nanmean).dropna()

In [12]:
dep_score

,weighted_deprivation_score
DHSID,
AL201700000001,0.019204
AL201700000002,0.009615
AL201700000003,0.000000
AL201700000004,0.024024
AL201700000005,0.004274
...,...
ZM201800000541,0.178197
ZM201800000542,0.367798
ZM201800000543,0.284661


## Import IWI data

In [13]:
#iwi data
iwi = pd.read_csv(r'C:\HDRO\Stats\Stats2\MOSAIKS\mean_IWI.csv')
iwi = iwi.groupby("DHSID")[["IWI"]].mean()

In [14]:
iwi

,IWI
DHSID,
AL201700000001,89.465391
AL201700000002,86.485039
AL201700000003,87.642180
AL201700000004,86.776051
AL201700000005,89.517279
...,...
ZW201500000396,38.506412
ZW201500000397,41.553372
ZW201500000398,31.797095


## join all data

In [15]:
merge1 = pd.merge(df4, nl, on=['DHSID'], how="inner")

In [16]:
merge2 = pd.merge(merge1, income, on=['DHSID'], how="inner")
merge3 = pd.merge(merge2, dep_score, on=['DHSID'], how="inner")
#merge4 = pd.merge(merge3, iwi, on=['DHSID'], how="inner")

In [17]:
merge3

,DHSID,X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,...,perc_pixels_in_bin_12,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score
0,AL201700000001,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.000000,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,18942.701172,0.019204
1,AL201700000002,0.218926,0.439225,0.103380,0.159651,0.315144,0.711148,0.225437,0.287943,0.098494,...,0.000000,0.108216,0.201556,0.110626,0.306916,0.000000,0.000000,0.000000,18275.437500,0.009615
2,AL201700000003,0.239452,0.515465,0.107474,0.204323,0.362553,0.710463,0.200439,0.286575,0.131596,...,0.000000,0.079555,0.229501,0.081326,0.306956,0.000000,0.000000,0.000000,19139.130859,0.000000
3,AL201700000004,0.230399,0.462435,0.107077,0.179146,0.332637,0.714292,0.223447,0.293186,0.113443,...,0.000000,0.093172,0.173535,0.095246,0.359493,0.052875,0.000000,0.000000,20030.707031,0.024024
4,AL201700000005,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.000000,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,22517.060547,0.004274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,RW201900000496,0.108982,0.350792,0.034645,0.137668,0.199491,0.341322,0.128825,0.150607,0.096750,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1787.015991,0.287234
52634,RW201900000497,0.138619,0.435800,0.039662,0.223957,0.251421,0.378330,0.129759,0.135954,0.172089,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1555.878784,0.369976
52635,RW201900000498,0.076103,0.296412,0.011021,0.264386,0.153407,0.179054,0.067392,0.056701,0.228597,...,0.015972,0.000000,0.015896,0.074712,0.000000,0.247376,0.140741,0.441666,3548.524414,0.210101
52636,RW201900000499,0.149145,0.475971,0.049168,0.185751,0.274041,0.468409,0.161029,0.185322,0.124372,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3598.289551,0.247942


In [18]:
#get the log of income
merge3['loginc'] = np.log(merge3['lognormal_inc_ind_wiid'])

In [19]:
#checking for missing
merge3.isnull().sum()

DHSID                         0
X_0                           0
X_1                           0
X_2                           0
X_3                           0
                             ..
perc_pixels_in_bin_18         0
perc_pixels_in_bin_19         0
lognormal_inc_ind_wiid        0
weighted_deprivation_score    0
loginc                        0
Length: 4024, dtype: int64

In [20]:
#getting the ID
merge3['ID']= merge3['DHSID'].str.slice(0,2)

# #getiing the numeric id
# merge4['ID_n'] = merge4['ID'].factorize()[0] + 1

# #moving the id in the front
# merge4.insert(1, 'ID_n1', merge4['ID_n'])

In [21]:
merge3.insert(1, 'country', merge3['ID'])
merge4 = merge3.drop(['DHSID', "ID"], axis=1)
merge4

,country,X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,...,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score,loginc
0,AL,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,18942.701172,0.019204,9.849174
1,AL,0.218926,0.439225,0.103380,0.159651,0.315144,0.711148,0.225437,0.287943,0.098494,...,0.108216,0.201556,0.110626,0.306916,0.000000,0.000000,0.000000,18275.437500,0.009615,9.813313
2,AL,0.239452,0.515465,0.107474,0.204323,0.362553,0.710463,0.200439,0.286575,0.131596,...,0.079555,0.229501,0.081326,0.306956,0.000000,0.000000,0.000000,19139.130859,0.000000,9.859490
3,AL,0.230399,0.462435,0.107077,0.179146,0.332637,0.714292,0.223447,0.293186,0.113443,...,0.093172,0.173535,0.095246,0.359493,0.052875,0.000000,0.000000,20030.707031,0.024024,9.905022
4,AL,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,22517.060547,0.004274,10.022029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,RW,0.108982,0.350792,0.034645,0.137668,0.199491,0.341322,0.128825,0.150607,0.096750,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1787.015991,0.287234,7.488303
52634,RW,0.138619,0.435800,0.039662,0.223957,0.251421,0.378330,0.129759,0.135954,0.172089,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1555.878784,0.369976,7.349796
52635,RW,0.076103,0.296412,0.011021,0.264386,0.153407,0.179054,0.067392,0.056701,0.228597,...,0.000000,0.015896,0.074712,0.000000,0.247376,0.140741,0.441666,3548.524414,0.210101,8.174287
52636,RW,0.149145,0.475971,0.049168,0.185751,0.274041,0.468409,0.161029,0.185322,0.124372,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3598.289551,0.247942,8.188214


In [22]:
#FINAL DATA
merge4

,country,X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,...,perc_pixels_in_bin_13,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score,loginc
0,AL,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,18942.701172,0.019204,9.849174
1,AL,0.218926,0.439225,0.103380,0.159651,0.315144,0.711148,0.225437,0.287943,0.098494,...,0.108216,0.201556,0.110626,0.306916,0.000000,0.000000,0.000000,18275.437500,0.009615,9.813313
2,AL,0.239452,0.515465,0.107474,0.204323,0.362553,0.710463,0.200439,0.286575,0.131596,...,0.079555,0.229501,0.081326,0.306956,0.000000,0.000000,0.000000,19139.130859,0.000000,9.859490
3,AL,0.230399,0.462435,0.107077,0.179146,0.332637,0.714292,0.223447,0.293186,0.113443,...,0.093172,0.173535,0.095246,0.359493,0.052875,0.000000,0.000000,20030.707031,0.024024,9.905022
4,AL,0.247140,0.503159,0.114064,0.203914,0.353395,0.727176,0.218433,0.306100,0.130447,...,0.088234,0.235019,0.082250,0.393638,0.045660,0.000000,0.000000,22517.060547,0.004274,10.022029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,RW,0.108982,0.350792,0.034645,0.137668,0.199491,0.341322,0.128825,0.150607,0.096750,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1787.015991,0.287234,7.488303
52634,RW,0.138619,0.435800,0.039662,0.223957,0.251421,0.378330,0.129759,0.135954,0.172089,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1555.878784,0.369976,7.349796
52635,RW,0.076103,0.296412,0.011021,0.264386,0.153407,0.179054,0.067392,0.056701,0.228597,...,0.000000,0.015896,0.074712,0.000000,0.247376,0.140741,0.441666,3548.524414,0.210101,8.174287
52636,RW,0.149145,0.475971,0.049168,0.185751,0.274041,0.468409,0.161029,0.185322,0.124372,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3598.289551,0.247942,8.188214


In [23]:
unique_values = merge4['country'].unique()

In [24]:
unique_values

array(['AL', 'AM', 'AO', 'BJ', 'BU', 'CM', 'EG', 'ET', 'GA', 'GN', 'GU',
       'HT', 'JO', 'KE', 'KM', 'LB', 'ML', 'MM', 'NG', 'PH', 'PK', 'SL',
       'SN', 'TJ', 'TL', 'TZ', 'UG', 'ZA', 'ZM', 'BF', 'GM', 'IA', 'KH',
       'MR', 'MZ', 'NI', 'RW'], dtype=object)

## within country analysis

In [25]:
# define a function to demean columns
def demean_column(grouped_col):
    return grouped_col - grouped_col.mean()

# group by the 'country' column and apply the demean function to each column
demeaned_df = merge4.iloc[:, 2:4025].groupby(merge4['country']).apply(demean_column)

#joining the countyr column back
demeaned_df = demeaned_df.join(merge4['country'])


In [26]:
demeaned_df

,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,...,perc_pixels_in_bin_14,perc_pixels_in_bin_15,perc_pixels_in_bin_16,perc_pixels_in_bin_17,perc_pixels_in_bin_18,perc_pixels_in_bin_19,lognormal_inc_ind_wiid,weighted_deprivation_score,loginc,country
0,0.084621,0.016802,0.034896,0.050608,0.057155,0.021160,-0.029396,0.021411,0.051269,0.051879,...,0.206662,0.057360,0.356938,-0.002086,-0.041036,-0.124213,5244.280273,-0.030550,0.437517,AL
1,0.020687,0.006118,-0.009367,0.012356,0.041126,0.028163,-0.047553,-0.010542,0.001740,0.007044,...,0.173199,0.085735,0.270216,-0.047746,-0.041036,-0.124213,4577.016602,-0.040139,0.401657,AL
2,0.096926,0.010211,0.035305,0.059765,0.040441,0.003166,-0.048921,0.022561,0.033992,0.042326,...,0.201144,0.056436,0.270256,-0.047746,-0.041036,-0.124213,5440.709961,-0.049755,0.447834,AL
3,0.043896,0.009814,0.010128,0.029849,0.044270,0.026173,-0.042310,0.004407,0.017624,0.026070,...,0.145179,0.070356,0.322793,0.005129,-0.041036,-0.124213,6332.286133,-0.025731,0.493365,AL
4,0.084621,0.016802,0.034896,0.050608,0.057155,0.021160,-0.029396,0.021411,0.051269,0.051879,...,0.206662,0.057360,0.356938,-0.002086,-0.041036,-0.124213,8818.639648,-0.045481,0.610373,AL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52633,-0.040447,0.001694,-0.118003,-0.024413,0.025389,0.023106,0.032358,-0.109825,-0.013265,-0.081719,...,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477,-399.525513,-0.025489,-0.032187,RW
52634,0.044561,0.006710,-0.031714,0.027516,0.062397,0.024040,0.017706,-0.034486,0.037715,0.004387,...,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477,-630.662720,0.057254,-0.170693,RW
52635,-0.094827,-0.021930,0.008714,-0.070498,-0.136880,-0.038327,-0.061547,0.022022,-0.098437,-0.042384,...,0.010807,0.067275,-0.004712,0.238135,0.131268,0.403188,1361.982910,-0.102622,0.653798,RW
52636,0.084732,0.016217,-0.069920,0.050136,0.152476,0.055310,0.067074,-0.082203,0.089818,-0.000830,...,-0.005089,-0.007437,-0.004712,-0.009241,-0.009473,-0.038477,1411.748047,-0.064780,0.667725,RW


## MOSAIKS model

In [41]:
#assign variables
data = demeaned_df.values

#select X features
X = data[:, 0:3999]

#create empty cell for resutls
results = []


# loop through each Y variable
for y_var in ['weighted_deprivation_score', 'loginc']:
    
    # select the current Y variable
    y = demeaned_df[y_var]
    
    # divide data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
    
    # define model evaluation method (k-fold crosss validation model)
    cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=111)

    # define model
    model = RidgeCV(alphas=[100, 10, 1, 0.1, 0.001, 0.0005], cv=cv)

    # fit model
    model.fit(X_train, y_train)
    
    # summarize chosen configuration
    print('alpha: %f' % model.alpha_)

    # predict model
    ridge_y_pred = model.predict(X_test)

    # get R^2 from true and predicted values
    #print('r2: %f' % r2_score(y_test,ridge_y_pred))
    #print('mse: %f' % mean_squared_error(y_test, ridge_y_pred))
    
    # store the test set predictions and results in a dictionary
    result = {'y_var': y_var, 'R2': r2_score(y_test,ridge_y_pred)}
    
    # append the result to the results list
    results.append(result)
    
# print the results
for result in results:
    print(result)

alpha: 0.001000
alpha: 0.001000
{'y_var': 'weighted_deprivation_score', 'R2': 0.47384611072986627}
{'y_var': 'loginc', 'R2': 0.5548209425226911}


## mosaiks + NL

In [44]:
#assign variables
data = demeaned_df.values

#select X features
X = data[:, 0:4019]

#create empty cell for resutls
results = []


# loop through each Y variable
for y_var in ['weighted_deprivation_score', 'loginc']:
    
    # select the current Y variable
    y = demeaned_df[y_var]
    
    # divide data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
    
    # define model evaluation method (k-fold crosss validation model)
    cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=111)

    # define model
    model = RidgeCV(alphas=[100, 10, 1, 0.1, 0.001, 0.0005], cv=cv)

    # fit model
    model.fit(X_train, y_train)
    
    # summarize chosen configuration
    print('alpha: %f' % model.alpha_)

    # predict model
    ridge_y_pred = model.predict(X_test)

    # get R^2 from true and predicted values
    #print('r2: %f' % r2_score(y_test,ridge_y_pred))
    #print('mse: %f' % mean_squared_error(y_test, ridge_y_pred))
    
    # store the test set predictions and results in a dictionary
    result = {'y_var': y_var, 'R2': r2_score(y_test,ridge_y_pred)}
    
    # append the result to the results list
    results.append(result)
    
# print the results
for result in results:
    print(result)

alpha: 0.001000
alpha: 0.001000
{'y_var': 'weighted_deprivation_score', 'R2': 0.5233437083893948}
{'y_var': 'loginc', 'R2': 0.6097099301503482}


## NL model

In [38]:
#assign variables
data = demeaned_df.values

#select X features
X = data[:, 4000:4019]

#create empty cell for resutls
results = []


# loop through each Y variable
for y_var in ['weighted_deprivation_score', 'loginc']:
    
    # select the current Y variable
    y = demeaned_df[y_var]
    
    # divide data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
    
    # define model evaluation method (k-fold crosss validation model)
    cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=111)

    # define model
    model = RidgeCV(alphas=[100, 10, 1, 0.1, 0.001, 0.0005], cv=cv)

    # fit model
    model.fit(X_train, y_train)
    
    # summarize chosen configuration
    print('alpha: %f' % model.alpha_)

    # predict model
    ridge_y_pred = model.predict(X_test)

    # get R^2 from true and predicted values
    #print('r2: %f' % r2_score(y_test,ridge_y_pred))
    #print('mse: %f' % mean_squared_error(y_test, ridge_y_pred))
    
    # store the test set predictions and results in a dictionary
    result = {'y_var': y_var, 'R2': r2_score(y_test,ridge_y_pred)}
    
    # append the result to the results list
    results.append(result)
    
# print the results
for result in results:
    print(result)

alpha: 10.000000
alpha: 10.000000
{'y_var': 'weighted_deprivation_score', 'R2': 0.38868929212976566}
{'y_var': 'loginc', 'R2': 0.5023855051211494}


In [ ]:
# alpha: 0.100000
# r2: 0.516376
# mse: 0.007486
# alpha: 0.100000
# r2: 0.577908
# mse: 97.711466
# alpha: 0.100000
# r2: 0.575334
# mse: 0.196135
# {'y_var': 'weighted_deprivation_score', 'R2': 0.5163755146443219}
# {'y_var': 'IWI', 'R2': 0.5779082013006809}
# {'y_var': 'loginc', 'R2': 0.5753339703939799}

## by country

## MOSAIKS model

In [43]:

#create empty cell for resutls
results = []

# loop through each Y variable
for y_var in ['weighted_deprivation_score', 'loginc']:
    
    # loop through each country
    for c in merge4['country'].unique():
        # select rows corresponding to the current country
        df_country = merge4[merge4['country'] == c]
        
        # separate X and y variables
        X = df_country.iloc[:, 1:4000]
        y = df_country[y_var]
        
        # divide data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

        # define model evaluation method (k-fold crosss validation model)
        cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=111)

        model = RidgeCV(alphas=[100, 10, 1, 0.1, 0.001, 0.0005], cv=cv)

        # fit model
        model.fit(X_train, y_train)

        # predict model
        ridge_y_pred = model.predict(X_test)

        # get R^2 from true and predicted values
        #print('r2: %f' % r2_score(y_test,ridge_y_pred))
        #print('mse: %f' % mean_squared_error(y_test, ridge_y_pred))

        # store the test set predictions and results in a dictionary
        result = {'country': c, 'y_var': y_var, 'R2': r2_score(y_test,ridge_y_pred)}

        # append the result to the results list
        results.append(result)

    # print the results
    for result in results:
        print(result)

{'country': 'AL', 'y_var': 'weighted_deprivation_score', 'R2': 0.3761935169968358}
{'country': 'AM', 'y_var': 'weighted_deprivation_score', 'R2': 0.29659801080400305}
{'country': 'AO', 'y_var': 'weighted_deprivation_score', 'R2': 0.4479657201089411}
{'country': 'BJ', 'y_var': 'weighted_deprivation_score', 'R2': 0.6456174529757193}
{'country': 'BU', 'y_var': 'weighted_deprivation_score', 'R2': 0.6065143696814597}
{'country': 'CM', 'y_var': 'weighted_deprivation_score', 'R2': 0.6201495817870215}
{'country': 'EG', 'y_var': 'weighted_deprivation_score', 'R2': 0.19889245326243632}
{'country': 'ET', 'y_var': 'weighted_deprivation_score', 'R2': 0.489059080619361}
{'country': 'GA', 'y_var': 'weighted_deprivation_score', 'R2': 0.5816506161704924}
{'country': 'GN', 'y_var': 'weighted_deprivation_score', 'R2': 0.6319692224100022}
{'country': 'GU', 'y_var': 'weighted_deprivation_score', 'R2': 0.4856203101941756}
{'country': 'HT', 'y_var': 'weighted_deprivation_score', 'R2': 0.5737853249519042}
{'c

## MOSAIKS + NL

In [46]:

#create empty cell for resutls
results = []

# loop through each Y variable
for y_var in ['weighted_deprivation_score', 'loginc']:
    
    # loop through each country
    for c in merge4['country'].unique():
        # select rows corresponding to the current country
        df_country = merge4[merge4['country'] == c]
        
        # separate X and y variables
        X = df_country.iloc[:, 1:4020]
        y = df_country[y_var]
        
        # divide data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

        # define model evaluation method (k-fold crosss validation model)
        cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=111)

        model = RidgeCV(alphas=[100, 10, 1, 0.1, 0.001, 0.0005], cv=cv)

        # fit model
        model.fit(X_train, y_train)

        # predict model
        ridge_y_pred = model.predict(X_test)

        # get R^2 from true and predicted values
        #print('r2: %f' % r2_score(y_test,ridge_y_pred))
        #print('mse: %f' % mean_squared_error(y_test, ridge_y_pred))

        # store the test set predictions and results in a dictionary
        result = {'country': c, 'y_var': y_var, 'R2': r2_score(y_test,ridge_y_pred)}

        # append the result to the results list
        results.append(result)

    # print the results
    for result in results:
        print(result)

{'country': 'AL', 'y_var': 'weighted_deprivation_score', 'R2': 0.3912322450082001}
{'country': 'AM', 'y_var': 'weighted_deprivation_score', 'R2': 0.4720990202529165}
{'country': 'AO', 'y_var': 'weighted_deprivation_score', 'R2': 0.6841850053645314}
{'country': 'BJ', 'y_var': 'weighted_deprivation_score', 'R2': 0.659907353983797}
{'country': 'BU', 'y_var': 'weighted_deprivation_score', 'R2': 0.626210018640276}
{'country': 'CM', 'y_var': 'weighted_deprivation_score', 'R2': 0.6682612463216023}
{'country': 'EG', 'y_var': 'weighted_deprivation_score', 'R2': 0.19931408078396717}
{'country': 'ET', 'y_var': 'weighted_deprivation_score', 'R2': 0.5375548391188669}
{'country': 'GA', 'y_var': 'weighted_deprivation_score', 'R2': 0.5964231499700392}
{'country': 'GN', 'y_var': 'weighted_deprivation_score', 'R2': 0.6329666374056432}
{'country': 'GU', 'y_var': 'weighted_deprivation_score', 'R2': 0.5518084661396419}
{'country': 'HT', 'y_var': 'weighted_deprivation_score', 'R2': 0.560900563018742}
{'coun

## NL

In [39]:

#create empty cell for resutls
results = []

# loop through each Y variable
for y_var in ['weighted_deprivation_score', 'loginc']:
    
    # loop through each country
    for c in merge4['country'].unique():
        # select rows corresponding to the current country
        df_country = merge4[merge4['country'] == c]
        
        # separate X and y variables
        X = df_country.iloc[:, 4000:4019]
        y = df_country[y_var]
        
        # divide data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

        # define model evaluation method (k-fold crosss validation model)
        cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=111)

        model = RidgeCV(alphas=[100, 10, 1, 0.1, 0.001, 0.0005], cv=cv)

        # fit model
        model.fit(X_train, y_train)

        # predict model
        ridge_y_pred = model.predict(X_test)

        # get R^2 from true and predicted values
        #print('r2: %f' % r2_score(y_test,ridge_y_pred))
        #print('mse: %f' % mean_squared_error(y_test, ridge_y_pred))

        # store the test set predictions and results in a dictionary
        result = {'country': c, 'y_var': y_var, 'R2': r2_score(y_test,ridge_y_pred)}

        # append the result to the results list
        results.append(result)

    # print the results
    for result in results:
        print(result)

{'country': 'AL', 'y_var': 'weighted_deprivation_score', 'R2': 0.2816601834584799}
{'country': 'AM', 'y_var': 'weighted_deprivation_score', 'R2': 0.22194196613288486}
{'country': 'AO', 'y_var': 'weighted_deprivation_score', 'R2': 0.6532720424514662}
{'country': 'BJ', 'y_var': 'weighted_deprivation_score', 'R2': 0.5666730361449055}
{'country': 'BU', 'y_var': 'weighted_deprivation_score', 'R2': 0.570228753696996}
{'country': 'CM', 'y_var': 'weighted_deprivation_score', 'R2': 0.5485170850146001}
{'country': 'EG', 'y_var': 'weighted_deprivation_score', 'R2': 0.13241462095408407}
{'country': 'ET', 'y_var': 'weighted_deprivation_score', 'R2': 0.5880991734795912}
{'country': 'GA', 'y_var': 'weighted_deprivation_score', 'R2': 0.5828563361411809}
{'country': 'GN', 'y_var': 'weighted_deprivation_score', 'R2': 0.5794455970036991}
{'country': 'GU', 'y_var': 'weighted_deprivation_score', 'R2': 0.4675216799359744}
{'country': 'HT', 'y_var': 'weighted_deprivation_score', 'R2': 0.3868435423348856}
{'c